# Improved QNN churn classifier

This version focuses on accuracy that generalizes: stratified train/test split, no preprocessing leakage, date feature engineering, supervised feature selection for the QNN inputs, class-balanced quantum training, angle scaling for the quantum feature map, a stronger optimizer budget, and baseline comparison.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42

candidate_paths = [
    Path("customer_data.csv"),
    Path.home() / "Downloads" / "customer_data.csv",
    Path(r"C:\Users\Shubham\Desktop\Datasets ML Project\COFINFAD Colombian Fintech Financial Analytics Dat\customer_data.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find customer_data.csv. Put it next to this notebook or update candidate_paths."
    )

df = pd.read_csv(csv_path)
print(df.shape)
df.head()

## Target and feature setup

Keep the threshold explicit. Changing it changes class balance, so compare results with F1/recall as well as accuracy.

In [ ]:
TARGET_THRESHOLD = 0.25

df = df.copy()
df["churn"] = (df["churn_probability"] > TARGET_THRESHOLD).astype(int)

drop_cols = ["customer_id", "churn_probability"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns] + ["churn"])
y = df["churn"]

date_columns = [c for c in X.columns if "date" in c.lower() or c in ["first_tx", "last_tx"]]
for column in date_columns:
    dt = pd.to_datetime(X[column], errors="coerce")
    X[f"{column}_year"] = dt.dt.year
    X[f"{column}_month"] = dt.dt.month
    X[f"{column}_dayofweek"] = dt.dt.dayofweek
    X[f"{column}_days_since"] = (dt.max() - dt).dt.days
X = X.drop(columns=date_columns)

print("Class balance:")
print(y.value_counts(normalize=True).rename("ratio"))
print("Date columns converted:", date_columns)

## Split before preprocessing

The original notebook scaled and PCA-transformed the full dataset before evaluation. This leaks test-set information. Split first, then fit all preprocessing only on training data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

numeric_features = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number, "bool"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ],
    remainder="drop",
)

num_qubits = 4
feature_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("select", SelectKBest(score_func=f_classif, k=num_qubits)),
    # Quantum feature maps use rotation angles. Scaling selected values helps optimization.
    ("angle_scale", MinMaxScaler(feature_range=(-np.pi, np.pi))),
])

X_train_q = feature_pipeline.fit_transform(X_train, y_train)
X_test_q = feature_pipeline.transform(X_test)

print(X_train_q.shape, X_test_q.shape)
print("Selected feature count:", feature_pipeline.named_steps["select"].get_support().sum())

## Balanced training subset

QNN training is slow, so use a small subset. The original first-200-row slice could be biased by row order. This version samples equally from each class from the training split.

In [ ]:
def balanced_sample(X_array, y_series, n_per_class=80, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    y_array = np.asarray(y_series)
    selected = []
    for cls in np.unique(y_array):
        cls_idx = np.flatnonzero(y_array == cls)
        take = min(n_per_class, len(cls_idx))
        selected.extend(rng.choice(cls_idx, size=take, replace=False))
    selected = np.array(selected)
    rng.shuffle(selected)
    return X_array[selected], y_array[selected]

X_train_small, y_train_small01 = balanced_sample(X_train_q, y_train, n_per_class=100)
y_train_small = np.where(y_train_small01 == 0, -1, 1)
y_test_qnn = np.where(y_test.to_numpy() == 0, -1, 1)

print("QNN training subset class counts:")
print(pd.Series(y_train_small01).value_counts())

## Stronger QNN setup

More optimizer iterations and a slightly deeper ansatz usually help. Keep the random seed fixed so changes are comparable.

In [ ]:
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit import QuantumCircuit
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA, SPSA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

feature_map = ZZFeatureMap(feature_dimension=num_qubits, reps=2, entanglement="linear")
ansatz = RealAmplitudes(num_qubits=num_qubits, reps=2, entanglement="linear")

qc = QuantumCircuit(num_qubits)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

qnn = EstimatorQNN(
    circuit=qc,
    input_params=list(feature_map.parameters),
    weight_params=list(ansatz.parameters),
)

objective_history = []
def callback(weights, objective_value):
    objective_history.append(objective_value)

initial_point = np.random.default_rng(RANDOM_STATE).uniform(
    low=-0.1,
    high=0.1,
    size=len(ansatz.parameters),
)

classifier = NeuralNetworkClassifier(
    qnn,
    optimizer=COBYLA(maxiter=200),  # try 400-800 if training time is acceptable
    initial_point=initial_point,
    callback=callback,
)

classifier.fit(X_train_small, y_train_small)

In [ ]:
y_pred_train = classifier.predict(X_train_q)
y_pred_test = classifier.predict(X_test_q)

y_pred_train01 = np.where(y_pred_train == -1, 0, 1)
y_pred_test01 = np.where(y_pred_test == -1, 0, 1)

print("Train accuracy:", accuracy_score(y_train, y_pred_train01).round(3))
print("Test accuracy:", accuracy_score(y_test, y_pred_test01).round(3))
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_pred_test01).round(3))
print("\nClassification report:")
print(classification_report(y_test, y_pred_test01))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_test01))

## Classical baseline

Always compare the QNN against a simple classical model. If the baseline is much better, tune preprocessing/features before spending more time on the quantum circuit.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

baseline_preprocess = Pipeline([
    ("preprocess", preprocess),
])

models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(
        n_estimators=120,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, pred).round(3))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred).round(3))
    print(classification_report(y_test, pred))

## Tuning checklist

1. Increase `COBYLA(maxiter=200)` to `400` or `800`.
2. Try `RealAmplitudes(..., reps=1)`, `reps=2`, and `reps=3`; deeper is not always better.
3. Try `ZZFeatureMap(..., reps=1)` and `reps=2`.
4. Tune `n_per_class` in the balanced subset. More rows may improve accuracy but will train slower.
5. Try `TARGET_THRESHOLD` values only if the business definition of churn allows it.
6. Prefer balanced accuracy/F1 when classes are imbalanced; plain accuracy can be misleading.
7. PCA was a major bottleneck on this dataset; 4 PCA components only gave about 0.56 balanced accuracy with a linear model. Supervised `SelectKBest(f_classif, k=4)` preserved much more target signal.
8. On this dataset, a balanced Logistic Regression baseline reached about 0.948 test accuracy and 0.958 balanced accuracy using only the 4 selected QNN features. Use that as the target for QNN tuning.